# Exercise 03 — Extending the LIF Simulation

**Module 03 | Estimated time: 1.5–3 hours**

---

## Overview

The basic LIF simulation from Lecture 2 uses constant input currents. Real neurons receive:
1. **Noisy inputs** — background synaptic bombardment modelled as Gaussian noise
2. **Stimulus-driven inputs** — a time-varying current step at t=200 ms
3. **Two neuron populations** — 80% excitatory (E), 20% inhibitory (I)

## Tasks

1. Add Gaussian noise to the input current at each timestep
2. Add a population-level stimulus current step
3. Separate excitatory and inhibitory neurons and colour the raster accordingly
4. *(Challenge)* Implement the Ornstein-Uhlenbeck (OU) noise process for biologically realistic fluctuations

**OU process:** $\tau_{noise} \, d\xi = -\xi \, dt + \sigma \sqrt{2 \tau_{noise}} \, dW$

Discretised: $\xi_{t+1} = \xi_t + \frac{dt}{\tau_{noise}}(-\xi_t) + \sigma \sqrt{2 dt / \tau_{noise}} \cdot \mathcal{N}(0,1)$

In [ ]:
!nvidia-smi

In [ ]:
%%writefile lif_noisy.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <curand_kernel.h>   // CUDA random number generation

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_dt, c_tau_m, c_E_L, c_Rm, c_V_th, c_V_reset;
__constant__ int   c_T_ref;
__constant__ float c_sigma;    // noise amplitude
__constant__ float c_I_stim;   // stimulus current added during [t_on, t_off]

// ── TODO 1: Initialize cuRAND states (one per neuron) ─────────────────────────
// curand_init(seed, sequence, offset, &state) initializes one RNG state.
// Each neuron should get a unique sequence number (use neuron index i).
__global__ void init_rng(curandState* states, int N, unsigned long seed) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    // TODO: call curand_init to initialize states[i]
    ???
}

// ── TODO 2: LIF step with noise ───────────────────────────────────────────────
// Modify the update equation to add Gaussian noise each step:
//   dV += sigma * curand_normal(&states[i]) * sqrt(dt)
// Also add c_I_stim if apply_stim is true (stimulus is active)
__global__ void lif_noisy_step(
    float* V, const float* I_bias, int* ref,
    int* spike_id, float* spike_t, int* n_spikes, int max_spikes,
    curandState* states,
    int N, float t_ms, int apply_stim
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    if (ref[i] > 0) { ref[i]--; V[i] = c_V_reset; return; }

    // TODO: compute the noisy input current
    float I_total = I_bias[i];
    if (apply_stim) I_total += ???;   // add stimulus
    // TODO: add noise term: sigma * curand_normal(&states[i]) * sqrt(dt)
    float noise = ???;

    float dV = c_dt / c_tau_m * (-(V[i] - c_E_L) + c_Rm * I_total) + noise;
    V[i] += dV;

    if (V[i] >= c_V_th) {
        V[i] = c_V_reset;
        ref[i] = c_T_ref;
        int idx = atomicAdd(n_spikes, 1);
        if (idx < max_spikes) { spike_id[idx] = i; spike_t[idx] = t_ms; }
    }
}

__global__ void init_state_ei(float* V, float* I_bias, int* ref, int N,
                               float I_E, float I_I) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    V[i]   = c_E_L;
    ref[i] = 0;
    // TODO Part 3: first 80% of neurons are excitatory (I_E), rest inhibitory (I_I)
    I_bias[i] = (i < (int)(0.8f * N)) ? ??? : ???;
}

int main() {
    const int N = 10000;
    const float T_ms = 1000.0f, dt = 0.1f;
    const float sigma = 0.5f;     // noise amplitude (mV/sqrt(ms))
    const float I_stim = 1.0f;    // extra stimulus current (pA)
    const float t_on = 200.0f, t_off = 600.0f;  // stimulus window (ms)

    float tau_m=20.f, E_L=-65.f, Rm=10.f, V_th=-55.f, V_reset=-70.f;
    int T_ref = (int)(2.0f/dt);
    int T_steps = (int)(T_ms/dt);

    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,      &dt,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_m,   &tau_m,   sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_L,     &E_L,     sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_Rm,      &Rm,      sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_th,    &V_th,    sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_reset, &V_reset, sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_T_ref,   &T_ref,   sizeof(int)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_sigma,   &sigma,   sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_I_stim,  &I_stim,  sizeof(float)));

    float *d_V, *d_I; int *d_ref, *d_sid, *d_ns; float *d_st;
    curandState* d_rng;
    size_t fb=N*sizeof(float), ib=N*sizeof(int);
    int max_spikes = N * 300;

    CUDA_CHECK(cudaMalloc(&d_V, fb)); CUDA_CHECK(cudaMalloc(&d_I, fb));
    CUDA_CHECK(cudaMalloc(&d_ref, ib));
    CUDA_CHECK(cudaMalloc(&d_sid, max_spikes*sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_st,  max_spikes*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_ns,  sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_rng, N*sizeof(curandState)));
    CUDA_CHECK(cudaMemset(d_ns, 0, sizeof(int)));

    int thr=256, blk=(N+thr-1)/thr;

    // TODO: call init_rng and init_state_ei
    ???
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));

    for (int step = 0; step < T_steps; step++) {
        float t_ms_now = step * dt;
        int stim = (t_ms_now >= t_on && t_ms_now < t_off) ? 1 : 0;
        lif_noisy_step<<<blk, thr>>>(
            d_V, d_I, d_ref,
            d_sid, d_st, d_ns, max_spikes,
            d_rng, N, t_ms_now, stim
        );
    }

    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    float ms; CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));

    int h_ns;
    CUDA_CHECK(cudaMemcpy(&h_ns, d_ns, sizeof(int), cudaMemcpyDeviceToHost));
    h_ns = (h_ns < max_spikes) ? h_ns : max_spikes;

    int*   h_sid = (int*)malloc(h_ns*sizeof(int));
    float* h_st  = (float*)malloc(h_ns*sizeof(float));
    CUDA_CHECK(cudaMemcpy(h_sid, d_sid, h_ns*sizeof(int),   cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_st,  d_st,  h_ns*sizeof(float), cudaMemcpyDeviceToHost));

    FILE* f = fopen("spikes_noisy.txt", "w");
    for (int k=0;k<h_ns;k++) fprintf(f,"%d %.2f\n", h_sid[k], h_st[k]);
    fclose(f);

    printf("N=%d  GPU=%.1f ms  spikes=%d  mean_fr=%.1f Hz\n",
           N, ms, h_ns, (float)h_ns/N/(T_ms/1000.0f));

    // cleanup...
    return 0;
}

In [ ]:
# Link curand library
!nvcc -O2 -o lif_noisy lif_noisy.cu -lm -lcurand && ./lif_noisy

In [ ]:
# Visualize: raster coloured by E/I population, with stimulus epoch highlighted
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

N = 10000
spikes = pd.read_csv('spikes_noisy.txt', sep=' ', names=['neuron', 'time_ms'])

sample_E = np.random.choice(range(int(0.8*N)), size=150, replace=False)
sample_I = np.random.choice(range(int(0.8*N), N), size=50, replace=False)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# Raster
for nid in sample_E:
    sp = spikes[spikes['neuron']==nid]['time_ms']
    ax1.scatter(sp, [nid]*len(sp), s=0.3, c='steelblue', alpha=0.7)
for nid in sample_I:
    sp = spikes[spikes['neuron']==nid]['time_ms']
    ax1.scatter(sp, [nid]*len(sp), s=0.5, c='tomato', alpha=0.7)

# Stimulus epoch
ax1.axvspan(200, 600, alpha=0.1, color='yellow', label='Stimulus on')
ax1.set_ylabel('Neuron ID', fontsize=12)
ax1.set_title('Noisy LIF Network — Raster Plot (E=blue, I=red)', fontsize=13)

# PSTH
bins = np.arange(0, 1001, 10)
for pop, label, color in [('E', 'Excitatory', 'steelblue'), ('I', 'Inhibitory', 'tomato')]:
    ids = range(int(0.8*N)) if pop=='E' else range(int(0.8*N), N)
    pop_spikes = spikes[spikes['neuron'].isin(ids)]
    counts, _ = np.histogram(pop_spikes['time_ms'], bins=bins)
    n_pop = int(0.8*N) if pop=='E' else int(0.2*N)
    rate = counts / n_pop / 0.01
    ax2.plot(bins[:-1]+5, rate, color=color, alpha=0.8, linewidth=1.2, label=label)

ax2.axvspan(200, 600, alpha=0.1, color='yellow')
ax2.set_xlabel('Time (ms)', fontsize=12)
ax2.set_ylabel('Rate (Hz)', fontsize=12)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('raster_ei.png', dpi=150, bbox_inches='tight')
plt.show()

---
Check against [ex03_solution.ipynb](ex03_solution.ipynb) when done.